PHOBERT

In [1]:
!pip install transformers datasets underthesea torch
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 48.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Import Libary

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import random

In [3]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Load Data

In [4]:
df = pd.read_csv("/content/drive/MyDrive/DACN/data/data.csv")

print(df.head())

                      content label  start
0               Áo bao đẹp ạ!   POS      5
1                   Tuyệt vời   POS      5
2   2day ao khong giong trong   NEG      1
3  Mùi thơm,bôi lên da mềm da   POS      5
4            Vải đẹp, dày dặn   POS      5


# Xử lí data

In [5]:
# Mapping label
label_map = {
    "NEG": 0,
    "NEU": 1,
    "POS": 2
}

df["label"] = df["label"].map(label_map)

# Chỉ giữ 2 cột cần thiết
df = df[["content", "label"]]

# Xoá dòng bị NaN nếu có
df = df.dropna()

print(df.head())

                      content  label
0               Áo bao đẹp ạ!      2
1                   Tuyệt vời      2
2   2day ao khong giong trong      0
3  Mùi thơm,bôi lên da mềm da      2
4            Vải đẹp, dày dặn      2


In [6]:
import re

def normalize_text(text):
    text = text.lower()

    # thay từ lóng phổ biến
    slang_dict = {
        "khong": "không",
        "ko": "không",
        "k": "không",
        "hok": "không",
        "giong": "giống",
        "sp": "sản phẩm",
        "mik": "mình",
        "dc": "được",
        "đc": "được"
    }

    for k, v in slang_dict.items():
        text = text.replace(k, v)

    # bỏ ký tự lặp nhiều lần (đẹpppp → đẹp)
    text = re.sub(r'(.)\1+', r'\1', text)

    # bỏ ký tự đặc biệt
    text = re.sub(r'[^\w\s]', ' ', text)

    # chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["content"] = df["content"].apply(normalize_text)

# Data Splitting

In [7]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["content"],
    df["label"],
    test_size=0.1,
    random_state=42
)

## Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)

## Dataset Construction

In [10]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": val_labels.tolist()
})

## Model Initialization

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base",
    num_labels=3
)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

## Evaluation Metrics Definition

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }

## Train Model

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate = 2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    seed = 42,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

Step,Training Loss
500,0.677800
1000,0.576000
1500,0.547157
2000,0.531396
2500,0.542880
3000,0.532566
3500,0.521385
4000,0.476104
4500,0.476151
5000,0.504197


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17685, training_loss=0.4367318181153034, metrics={'train_runtime': 4090.4942, 'train_samples_per_second': 34.583, 'train_steps_per_second': 4.323, 'total_flos': 8069184906191640.0, 'train_loss': 0.4367318181153034, 'epoch': 5.0})

In [17]:
pred = trainer.predict(val_dataset)
print(pred.metrics)

{'test_loss': 0.6809000372886658, 'test_accuracy': 0.7993002544529262, 'test_f1': 0.7920621920500237, 'test_runtime': 18.4798, 'test_samples_per_second': 170.132, 'test_steps_per_second': 21.266}


## Dự đoán

In [18]:
def predict(text):

    text = normalize_text(text)   # chuẩn hóa câu

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    reverse_label_map = {
        0: "NEG",
        1: "NEU",
        2: "POS"
    }

    return reverse_label_map[predicted_class]

In [20]:
print(predict("Sản phẩm này rất tốt"))
print(predict("không giống ảnh "))
print(predict("mắc rất khó chịu"))
print(predict("tạm chấp nhận đc nhưng k ấn tượng lắm"))
print(predict("woah"))
print(predict("sương sương"))
print(predict("quá tệ"))
print(predict("woah, k thể tin nỗi là sp quá tệ"))
print(predict("Sp này mik cảm thấy k nên mua"))

POS
NEG
NEG
NEU
POS
NEU
NEG
NEG
NEG


## Lưu Model

In [22]:
trainer.save_model("/content/drive/MyDrive/DACN/PhoBert/best_model")
tokenizer.save_pretrained("/content/drive/MyDrive/DACN/PhoBert/best_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/DACN/PhoBert/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/DACN/PhoBert/best_model/vocab.txt',
 '/content/drive/MyDrive/DACN/PhoBert/best_model/bpe.codes',
 '/content/drive/MyDrive/DACN/PhoBert/best_model/added_tokens.json')